In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/paysim1/PS_20174392719_1491204439457_log.csv


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ealaxi/paysim1")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/paysim1


In [3]:
df = pd.read_csv('/kaggle/input/paysim1/PS_20174392719_1491204439457_log.csv')
df.head(5)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## Dataset Understanding
* One row represents One transaction/payment
* Time granularity is for every transaction/payment
* Columns that appear to be:
  * User identifier(s): nameOrig, nameDest
  * Monetary value(s): amount, oldbalanceOrg, newbalanceOrg, oldbalanceDest, newbalanceDest
  * Time variable(s): step
  * Outcome / status / risk signal(s): isFraud, isFlaggedFraud

In [4]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='object')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [6]:
required_cols = ['step','type','amount','oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest','newbalanceDest','isFraud']

clean_df = df[required_cols].copy()
clean_df.head(5)

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0


In [7]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 8 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   oldbalanceOrg   float64
 4   newbalanceOrig  float64
 5   oldbalanceDest  float64
 6   newbalanceDest  float64
 7   isFraud         int64  
dtypes: float64(5), int64(2), object(1)
memory usage: 388.3+ MB


In [8]:
clean_df.isna().sum()

step              0
type              0
amount            0
oldbalanceOrg     0
newbalanceOrig    0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
dtype: int64

No Null Values

In [9]:
clean_df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00


## Balance Consistensy

In [10]:
Origin_check = clean_df['oldbalanceOrg'] - clean_df['amount'] == clean_df['newbalanceOrig']

if Origin_check.all():
    print("Origin Balance is Equal")
else:
    print("Origin Balance is Not Equal")


Dest_check = clean_df['oldbalanceDest'] + clean_df['amount'] == clean_df['newbalanceDest']
if Dest_check.all():
    print("Destination Balance is Equal")
else:
    print("Destination Balance is Not Equal")


Origin Balance is Not Equal
Destination Balance is Not Equal


In [11]:
count_org = (df['oldbalanceOrg'] - df['amount'] != df['newbalanceOrig']).sum()
print("Number of rows where Origin Balance inconsistency is there:", count_org)

count_dest = (df['oldbalanceDest'] + df['amount'] != df['newbalanceDest']).sum()
print("Number of rows where Dest Balance inconsistency is there:", count_dest)

Number of rows where Origin Balance inconsistency is there: 5413997
Number of rows where Dest Balance inconsistency is there: 4736674


In [12]:
per_count_org = (count_org/6362620)*100
print("Percentage of rows where Origin Balance inconsisteny:",per_count_org)

per_count_dest = (count_dest/6362620)*100
print("Percentage of rows where dest Balance inconsisteny",per_count_dest)

Percentage of rows where Origin Balance inconsisteny: 85.09068591240715
Percentage of rows where dest Balance inconsisteny 74.44533855550071


In [13]:
origin_check = df['oldbalanceOrg'] - df['amount'] != df['newbalanceOrig']
fra_bal_check_org = (origin_check & (df['isFraud'] == 1)).sum()
print("Origin Balance inconsistency related to fraud:", fra_bal_check_org)

Dest_check = df['oldbalanceDest'] + df['amount'] != df['newbalanceDest']
fra_bal_check_dest = (Dest_check & (df['isFraud'] == 1)).sum()
print("Dest Balance inconsistency related to fraud:", fra_bal_check_dest)

Origin Balance inconsistency related to fraud: 127
Dest Balance inconsistency related to fraud: 5324


In [14]:
per_count_org_fra = (fra_bal_check_org/count_org)*100
print("Percentage of rows where Origin Balance inconsisteny related to fraud:",per_count_org_fra)

per_count_dest_fra = (fra_bal_check_dest/count_dest)*100
print("Percentage of rows where dest Balance inconsisteny related to fraud",per_count_dest_fra)

Percentage of rows where Origin Balance inconsisteny related to fraud: 0.002345771525178163
Percentage of rows where dest Balance inconsisteny related to fraud 0.11239954449050113


## Impossible Transactions

In [15]:
if (df['amount'] < 0).any():
    print("Negative amount exists")
else:
    print("None")
    
if (df['oldbalanceOrg'] < 0).any():
    print("Negative oldbalanceOrg exists")
else:
    print("None")
    
if (df['newbalanceOrig'] < 0).any():
    print("Negative newbalanceOrig exists")
else:
    print("None")
    
if (df['oldbalanceDest'] < 0).any():
    print("Negative oldbalanceDest exists")
else:
    print("None")
    
if (df['newbalanceDest'] < 0).any():
    print("Negative newbalanceDest exists")
else:
    print("None")

if(df['amount']>df['oldbalanceOrg']).any():
    print("amount greater than balance exist")
else:
    print("None")


None
None
None
None
None
amount greater than balance exist


In [16]:
count = (df['amount']>df['oldbalanceOrg']).sum()
print("Number of rows where amount > oldbalanceOrg:", count)

Number of rows where amount > oldbalanceOrg: 4079080


In [17]:
per_count = (count/6362620)*100
print("Percentage of rows where amount > oldbalanceOrg:",per_count)

Percentage of rows where amount > oldbalanceOrg: 64.11006786512475


## Duplicates

In [18]:
clean_df.duplicated().sum()

np.int64(543)

In [19]:
clean_df = clean_df.drop_duplicates()
clean_df.duplicated().sum()

np.int64(0)

## USER-TIME BALANCE VIEW

In [20]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='object')

In [21]:
required_cols_2 = ['nameOrig', 'step', 'amount', 'newbalanceOrig','isFraud']

user_balance_df = df[required_cols_2].copy()
user_balance_df.head(5)

,nameOrig,step,amount,newbalanceOrig,isFraud
0,C1231006815,1,9839.64,160296.36,0
1,C1666544295,1,1864.28,19384.72,0
2,C1305486145,1,181.00,0.00,1
3,C840083671,1,181.00,0.00,1
4,C2048537720,1,11668.14,29885.86,0


In [22]:
user_balance_df.duplicated().sum()

np.int64(0)

In [23]:
user_balance_df.info

<bound method DataFrame.info of             nameOrig  step      amount  newbalanceOrig  isFraud
0        C1231006815     1     9839.64       160296.36        0
1        C1666544295     1     1864.28        19384.72        0
2        C1305486145     1      181.00            0.00        1
3         C840083671     1      181.00            0.00        1
4        C2048537720     1    11668.14        29885.86        0
...              ...   ...         ...             ...      ...
6362615   C786484425   743   339682.13            0.00        1
6362616  C1529008245   743  6311409.28            0.00        1
6362617  C1162922333   743  6311409.28            0.00        1
6362618  C1685995037   743   850002.52            0.00        1
6362619  C1280323807   743   850002.52            0.00        1

[6362620 rows x 5 columns]>

In [24]:
user_balance_df = user_balance_df.sort_values(['nameOrig','step'])

In [25]:
unique_users = user_balance_df['nameOrig'].nunique()
total_rows = len(user_balance_df)
per_uni_userid = (unique_users/total_rows)*100
print("Unique No.of users:", per_uni_userid)

Unique No.of users: 99.85362947967975


In [26]:
print("Total transactions:", total_rows)
print("Unique users:", unique_users)
print("Average transactions per user:", total_rows / unique_users)

Total transactions: 6362620
Unique users: 6353307
Average transactions per user: 1.0014658507766112
